# 쇼핑몰 DB CRUD 시연 — 회원 · 상품 · 주문

- 사전 준비: `sql/01_schema.sql` 실행, `.env` 작성 (README 참고)
- 위에서부터 순서대로 실행합니다. 각 셀 출력이 README의 CRUD 캡처가 됩니다.
- 판매자(SELLER)와 구매자(BUYER) 두 계정으로 RLS(사용자별 접근 제어)를 확인합니다.

In [ ]:
import os
from pprint import pprint

from services import MemberService, OrderService, ProductService, get_client, login

supabase = get_client()
members = MemberService(supabase)
products = ProductService(supabase)
orders = OrderService(supabase)
print("Supabase 연결 완료")

## 1. 회원(members) CRUD — 판매자 계정

In [ ]:
seller = login(supabase, os.getenv("SELLER_EMAIL"), os.getenv("SELLER_PASSWORD"))
print("로그인:", seller.email)

# [CREATE] 회원 정보 등록 (이미 있으면 기존 정보 사용)
me = members.get_me() or members.create(name="김판매", role="SELLER", phone="010-1111-2222")
print("[CREATE] 회원 등록")
pprint(me)

In [ ]:
# [READ] 내 회원 정보 조회
print("[READ] 내 정보")
pprint(members.get_me())

In [ ]:
# [UPDATE] 연락처·주소 수정 (updated_at 자동 갱신)
print("[UPDATE] 회원 정보 수정")
pprint(members.update(phone="010-3333-4444", address="서울시 강남구"))

## 2. 상품(products) CRUD — 판매자 계정

In [ ]:
# [CREATE] 상품 등록 (SELLER만 가능)
keyboard = products.create(name="무선 키보드", price=45000, stock=10, description="블루투스 5.0")
mouse = products.create(name="무선 마우스", price=25000, stock=20)
print("[CREATE] 상품 등록")
pprint([keyboard, mouse])

In [ ]:
# [READ] 상품 목록 / 검색
print("[READ] 전체 상품")
pprint(products.list())
print("[READ] '키보드' 검색")
pprint(products.list("키보드"))

In [ ]:
# [UPDATE] 가격·재고 수정 (본인 상품만)
print("[UPDATE] 상품 수정")
pprint(products.update(keyboard["id"], price=39000, stock=15))

In [ ]:
# [DELETE] 상품 삭제 (본인 상품만)
print("[DELETE] 상품 삭제")
pprint(products.delete(mouse["id"]))
print("삭제 후 조회:", products.get(mouse["id"]))

## 3. 주문(orders) CRUD — 구매자 계정

In [ ]:
supabase.auth.sign_out()
buyer = login(supabase, os.getenv("BUYER_EMAIL"), os.getenv("BUYER_PASSWORD"))
print("로그인:", buyer.email)
buyer_info = members.get_me() or members.create(name="이구매", role="BUYER", address="부산시 해운대구")
pprint(buyer_info)

In [ ]:
# [CREATE] 주문 생성 (주문 시점 가격 기록, total_price는 DB가 계산)
order = orders.create(product_id=keyboard["id"], quantity=2)
print("[CREATE] 주문 생성")
pprint(order)

In [ ]:
# [READ] 내 주문 목록 (상품명 JOIN)
print("[READ] 내 주문 목록")
pprint(orders.list_mine())

In [ ]:
# [UPDATE] 주문 상태 변경
print("[UPDATE] 주문 상태 변경")
pprint(orders.update_status(order["id"], "PAID"))

## 4. 접근 제어(RLS)와 예외 처리 확인 — 구매자 계정

아래 요청은 모두 **실패해야 정상**입니다.

In [ ]:
checks = {
    "구매자가 판매자 상품 가격 수정": lambda: products.update(keyboard["id"], price=1),
    "구매자가 판매자 상품 삭제": lambda: products.delete(keyboard["id"]),
    "BUYER 회원의 상품 등록": lambda: products.create(name="권한 없는 상품", price=1000),
    "수량 0 주문 (CHECK 제약)": lambda: orders.create(product_id=keyboard["id"], quantity=0),
    "없는 상품 주문": lambda: orders.create(product_id=999999, quantity=1),
    "허용되지 않은 주문 상태": lambda: orders.update_status(order["id"], "REFUNDED"),
}
for title, action in checks.items():
    try:
        action()
        print(f"[실패해야 함] {title}: 성공해 버림")
    except Exception as e:
        print(f"[차단됨] {title}\n    -> {type(e).__name__}: {e}")

## 5. 삭제(DELETE) — 주문 → 회원 순서 (주문이 있는 회원은 FK RESTRICT로 삭제 불가)

In [ ]:
# [DELETE] 주문이 남아 있으면 회원 삭제가 막히는지 먼저 확인
try:
    members.delete()
except Exception as e:
    print("[차단됨] 주문이 있는 회원 삭제 ->", e)

# [DELETE] 주문 삭제 후 회원 삭제
print("[DELETE] 주문 삭제")
pprint(orders.delete(order["id"]))
print("[DELETE] 회원 삭제")
pprint(members.delete())
print("삭제 후 내 정보:", members.get_me())

In [ ]:
supabase.auth.sign_out()
print("로그아웃 완료")